In [9]:
import os
import sys
import re
import json
import difflib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ── 项目根目录解析 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.cwe import CWEUtils
from src.utils.cve import CVEUtils
from src.utils.formatter import auto_format_c_code_batch, detect_c_or_cpp

# ── 超参数 ──
MIN_CODE_LEN       = 50       # 最短代码字符数
MAX_CODE_LEN       = 65535    # 最长代码字符数（Milvus VARCHAR 上限）

# MERGE_FUNCS_AS_CVE = False     # 是否按 (CVE ID, commit_id) 合并同源多函数
# OUTPUT_PATH        = PROJECT_ROOT / "data/raw/vul/ReposVul_v1.jsonl"

MERGE_FUNCS_AS_CVE = True     # 是否按 (CVE ID, commit_id) 合并同源多函数
OUTPUT_PATH        = PROJECT_ROOT / "data/raw/vul/ReposVul_Merged_v1.jsonl"

# ── C/C++ 文件扩展名白名单（以 details[].file_language 为准，非 cve_language）──
C_EXTENSIONS      = {'c'}
CPP_EXTENSIONS    = {'cpp', 'cc', 'cxx', 'c++'}
HEADER_EXTENSIONS = {'h', 'hpp', 'hh', 'hxx', 'h++'}
VALID_EXTENSIONS  = C_EXTENSIONS | CPP_EXTENSIONS | HEADER_EXTENSIONS

# ── 数据集路径 ──
DATASET_FILES = [
    Path("dataset/raw/ReposVul_c.jsonl"),
    Path("dataset/raw/ReposVul_cpp.jsonl"),
]

def _quick_valid(val):
    """轻量级的字符串非空判断。"""
    if not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != "none"

def _extract_first_cwe(cwe_field):
    """从 ReposVul 的 cwe_id 字段（list 或 str）中提取第一个 CWE ID。"""
    if isinstance(cwe_field, list) and len(cwe_field) > 0:
        return str(cwe_field[0]).strip()
    if isinstance(cwe_field, str):
        m = re.search(r"CWE-\d+", cwe_field, re.IGNORECASE)
        return m.group() if m else cwe_field.strip().strip("[]'\"")
    return None

# ── 流式加载 + 展平为函数级记录 ──
# ReposVul 每条 JSONL 记录是 commit 级，需展平为函数级 (vul_code, fixed_code) 对。
# 展平逻辑：遍历 details[]，以 file_language 过滤 C/C++ 文件，
# 然后按下标配对 function_before[i] 与 function_after[i]。
records = []
total_records = 0
skipped_outdated = 0
skipped_non_c_cpp = 0
skipped_no_funcs = 0

for fpath in DATASET_FILES:
    with open(fpath, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"展平 {fpath.name}"):
            record = json.loads(line)
            total_records += 1

            # 跳过 outdated 记录
            if record.get("outdated") == 1:
                skipped_outdated += 1
                continue

            cve_id = record.get("cve_id")
            cwe_id_raw = record.get("cwe_id")
            cve_description = record.get("cve_description", "")
            commit_id = record.get("commit_id", "")

            cwe_id = _extract_first_cwe(cwe_id_raw)

            for detail in record.get("details", []):
                file_lang = str(detail.get("file_language", "")).strip().lower()

                if file_lang not in VALID_EXTENSIONS:
                    skipped_non_c_cpp += 1
                    continue

                fb_list = detail.get("function_before") or []
                fa_list = detail.get("function_after") or []

                if not fb_list or not fa_list:
                    skipped_no_funcs += 1
                    continue

                pair_count = min(len(fb_list), len(fa_list))
                for idx in range(pair_count):
                    fb_item = fb_list[idx]
                    fa_item = fa_list[idx]

                    if not isinstance(fb_item, dict) or not isinstance(fa_item, dict):
                        continue

                    vul_code = fb_item.get("function", "")
                    fixed_code = fa_item.get("function", "")

                    if not _quick_valid(vul_code) or not _quick_valid(fixed_code):
                        continue
                    if vul_code == fixed_code:
                        continue
                    if not (MIN_CODE_LEN <= len(vul_code) <= MAX_CODE_LEN):
                        continue
                    if not (MIN_CODE_LEN <= len(fixed_code) <= MAX_CODE_LEN):
                        continue

                    records.append({
                        "cve_id": cve_id,
                        "cwe_id": cwe_id,
                        "cve_description": cve_description,
                        "commit_id": commit_id,
                        "file_language": file_lang,
                        "vul_code": vul_code,
                        "fixed_code": fixed_code,
                    })

df_raw = pd.DataFrame(records)
del records

print(f"\n原始 JSONL 记录数:        {total_records}")
print(f"  跳过 outdated:          {skipped_outdated}")
print(f"  跳过非 C/C++ 文件:      {skipped_non_c_cpp}")
print(f"  跳过无函数对的 detail:  {skipped_no_funcs}")
print(f"展平后函数级样本数:       {len(df_raw)}")
df_raw.head()

Project root: <PROJECT_ROOT>


展平 ReposVul_c.jsonl: 4010it [00:41, 96.47it/s] 
展平 ReposVul_cpp.jsonl: 689it [00:08, 85.83it/s] 


原始 JSONL 记录数:        4699
  跳过 outdated:          693
  跳过非 C/C++ 文件:      2222
  跳过无函数对的 detail:  5317
展平后函数级样本数:       7363


,cve_id,cwe_id,cve_description,commit_id,file_language,vul_code,fixed_code
0,CVE-2011-0990,CWE-362,Race condition in the FastCopy optimization in...,2f00e4bbb2137130845afb1b2a1e678552fc8e5c,c,static gboolean\nves_icall_System_Array_FastCo...,static gboolean\nves_icall_System_Array_FastCo...
1,CVE-2011-0991,CWE-399,"Use-after-free vulnerability in Mono, when Moo...",89d1455a80ef13cddee5d79ec00c06055da3085c,c,void\nmono_runtime_shutdown (void)\n{\n\tmono_...,void\nmono_runtime_shutdown (void)\n{\n\tmono_...
2,CVE-2011-0991,CWE-399,"Use-after-free vulnerability in Mono, when Moo...",3f8ee42b8c867d9a4c18c22657840d072cca5c3a,c,void\nmono_reflection_destroy_dynamic_method (...,gboolean\nmono_reflection_is_valid_dynamic_tok...
3,CVE-2011-0991,CWE-399,"Use-after-free vulnerability in Mono, when Moo...",3f8ee42b8c867d9a4c18c22657840d072cca5c3a,c,gboolean\nmono_reflection_is_valid_dynamic_tok...,static guint32\nmono_declsec_get_flags (MonoIm...
4,CVE-2011-0991,CWE-399,"Use-after-free vulnerability in Mono, when Moo...",3f8ee42b8c867d9a4c18c22657840d072cca5c3a,c,static guint32\nmono_declsec_get_flags (MonoIm...,guint32\nmono_declsec_flags_from_method (MonoM...


In [10]:
df = df_raw.copy()

def _is_valid_str(val):
    """判断值是否为非空、非 None 的字符串。"""
    if val is None:
        return False
    if not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != 'none'

# ── 步骤 1：vul_code 和 fixed_code 非空 ──
df = df[
    df['vul_code'].apply(_is_valid_str)
    & df['fixed_code'].apply(_is_valid_str)
]
print(f"步骤1 (代码非空):          {len(df):>7} 条")

# ── 步骤 2：vul_code != fixed_code（漏洞修复真实有效） ──
df = df[df['vul_code'] != df['fixed_code']]
print(f"步骤2 (vul!=fixed):        {len(df):>7} 条")

# ── 步骤 3：cwe_id 非空 ──
df = df[df['cwe_id'].apply(_is_valid_str)]
print(f"步骤3 (CWE非空):           {len(df):>7} 条")

# ── 步骤 4：cve_id 非空 ──
df = df[df['cve_id'].apply(_is_valid_str)]
print(f"步骤4 (CVE非空):           {len(df):>7} 条")

# ── 步骤 5：代码长度过滤 ──
vul_len   = df['vul_code'].str.len()
fixed_len = df['fixed_code'].str.len()
df = df[
    vul_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
    & fixed_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
]
print(f"步骤5 (代码长度合法):      {len(df):>7} 条")

df = df.reset_index(drop=True)
print(f"\n清洗后总样本数: {len(df)}")
df[['cwe_id', 'cve_id', 'file_language', 'vul_code', 'fixed_code']].head()

步骤1 (代码非空):             7363 条
步骤2 (vul!=fixed):           7363 条
步骤3 (CWE非空):              7363 条
步骤4 (CVE非空):              7363 条
步骤5 (代码长度合法):         7363 条

清洗后总样本数: 7363


,cwe_id,cve_id,file_language,vul_code,fixed_code
0,CWE-362,CVE-2011-0990,c,static gboolean\nves_icall_System_Array_FastCo...,static gboolean\nves_icall_System_Array_FastCo...
1,CWE-399,CVE-2011-0991,c,void\nmono_runtime_shutdown (void)\n{\n\tmono_...,void\nmono_runtime_shutdown (void)\n{\n\tmono_...
2,CWE-399,CVE-2011-0991,c,void\nmono_reflection_destroy_dynamic_method (...,gboolean\nmono_reflection_is_valid_dynamic_tok...
3,CWE-399,CVE-2011-0991,c,gboolean\nmono_reflection_is_valid_dynamic_tok...,static guint32\nmono_declsec_get_flags (MonoIm...
4,CWE-399,CVE-2011-0991,c,static guint32\nmono_declsec_get_flags (MonoIm...,guint32\nmono_declsec_flags_from_method (MonoM...


In [11]:
SEP = "\n\n/* ---- next function ---- */\n\n"

if MERGE_FUNCS_AS_CVE:
    print("启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...")

    merged_rows = []
    for (cve_id, commit_id), grp in df.groupby(['cve_id', 'commit_id'], sort=False):
        first = grp.iloc[0].to_dict()
        first['cve_id']     = cve_id
        first['commit_id']  = commit_id
        first['vul_code']   = SEP.join(grp['vul_code'].tolist())
        first['fixed_code'] = SEP.join(grp['fixed_code'].tolist())
        merged_rows.append(first)

    df = pd.DataFrame(merged_rows).reset_index(drop=True)

    # 合并后再次过滤超长代码
    df = df[
        df['vul_code'].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
        & df['fixed_code'].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
    ].reset_index(drop=True)

    print(f"合并后样本数: {len(df)}")
else:
    print(f"跳过同源 CVE 合并 (MERGE_FUNCS_AS_CVE=False)，当前样本数: {len(df)}")

df[['cve_id', 'cwe_id', 'file_language', 'vul_code', 'fixed_code']].head()

启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...
合并后样本数: 1131


,cve_id,cwe_id,file_language,vul_code,fixed_code
0,CVE-2011-0990,CWE-362,c,static gboolean\nves_icall_System_Array_FastCo...,static gboolean\nves_icall_System_Array_FastCo...
1,CVE-2011-0991,CWE-399,c,void\nmono_runtime_shutdown (void)\n{\n\tmono_...,void\nmono_runtime_shutdown (void)\n{\n\tmono_...
2,CVE-2011-0991,CWE-399,c,void\nmono_reflection_destroy_dynamic_method (...,gboolean\nmono_reflection_is_valid_dynamic_tok...
3,CVE-2011-2517,CWE-119,c,static int nl80211_trigger_scan(struct sk_buff...,static int nl80211_trigger_scan(struct sk_buff...
4,CVE-2011-2518,CWE-20,c,static int tomoyo_mount_acl(struct tomoyo_requ...,static int tomoyo_mount_acl(struct tomoyo_requ...


In [12]:
from src.utils.formatter import auto_format_c_code_batch, ClangFormatStyle

print("格式化 vul_code ...")
vul_results = auto_format_c_code_batch(
    df['vul_code'].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df['vul_code'] = [r[0] for r in vul_results]

print("格式化 fixed_code ...")
fixed_results = auto_format_c_code_batch(
    df['fixed_code'].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df['fixed_code'] = [r[0] for r in fixed_results]

low_conf_vul   = sum(1 for r in vul_results   if r[1].confidence < 0.6)
low_conf_fixed = sum(1 for r in fixed_results if r[1].confidence < 0.6)
print(f"\nvul_code   低置信度样本数 (<0.6): {low_conf_vul}")
print(f"fixed_code 低置信度样本数 (<0.6): {low_conf_fixed}")
print(f"格式化完成，总样本数: {len(df)}")

# ── 格式化后再次过滤 vul_code == fixed_code ──
# clang-format 会将仅存在空白符/缩进差异的代码对统一为相同字符串
before_dedup = len(df)
df = df[df['vul_code'] != df['fixed_code']].reset_index(drop=True)
removed = before_dedup - len(df)
print(f"\n格式化后去重 (vul!=fixed): 移除 {removed} 条，剩余 {len(df)} 条")

格式化 vul_code ...


Formatting: 100%|██████████| 1131/1131 [00:09<00:00, 121.87snippet/s]

格式化 fixed_code ...



Formatting: 100%|██████████| 1131/1131 [00:09<00:00, 113.71snippet/s]


vul_code   低置信度样本数 (<0.6): 28
fixed_code 低置信度样本数 (<0.6): 29
格式化完成，总样本数: 1131

格式化后去重 (vul!=fixed): 移除 2 条，剩余 1129 条


In [13]:
cwe_utils = CWEUtils(PROJECT_ROOT / "data/raw/cwe/cwec_v4.19.1.xml")

def get_cwe_desc(cwe_field):
    """将 cwe_id 字段转换为 CWEUtils prompt 文本。"""
    if cwe_field is None:
        return None
    cwe_id = None
    if isinstance(cwe_field, np.ndarray) and len(cwe_field) > 0:
        cwe_id = str(cwe_field[0])
    elif isinstance(cwe_field, list) and len(cwe_field) > 0:
        cwe_id = str(cwe_field[0])
    elif isinstance(cwe_field, str):
        m = re.search(r"CWE-\d+", cwe_field, re.IGNORECASE)
        cwe_id = m.group() if m else cwe_field.strip().strip("[]'\"").split(",")[0].strip().strip("'\"")
    if not cwe_id or not cwe_id.strip():
        return None
    prompt = cwe_utils.get_prompt(cwe_id.strip())
    if "not found in the CWE dataset" in prompt:
        return None
    return prompt

df['cwe_desc'] = df['cwe_id'].apply(get_cwe_desc)

filled = df['cwe_desc'].notna().sum()
print(f"总样本数:              {len(df)}")
print(f"cwe_desc 已填充:       {filled}")
print(f"cwe_desc 为空:         {len(df) - filled}")
print(f"cwe_desc 填充率:       {filled / len(df) * 100:.2f}%")
df[['cwe_id', 'cwe_desc']].head()

总样本数:              1129
cwe_desc 已填充:       1100
cwe_desc 为空:         29
cwe_desc 填充率:       97.43%


,cwe_id,cwe_desc
0,CWE-362,- CWE ID: CWE-362\n- Name: Concurrent Executio...
1,CWE-399,- CWE ID: CWE-399\n- Name: Resource Management...
2,CWE-399,- CWE ID: CWE-399\n- Name: Resource Management...
3,CWE-119,- CWE ID: CWE-119\n- Name: Improper Restrictio...
4,CWE-20,- CWE ID: CWE-20\n- Name: Improper Input Valid...


In [14]:
cve_utils = CVEUtils(PROJECT_ROOT / "data/raw/cwe/CVE_v2026.03.18.json")

def _normalize_cve_desc(val):
    """将 cve_description 字段归一化为字符串，过滤空值。"""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s if s and s.lower() != 'none' else None
    return None

def get_cve_desc(row):
    """CVE 描述获取策略：优先使用 ReposVul 原始 cve_description 字段，不足时用 CVEUtils 补充。"""
    raw_desc = _normalize_cve_desc(row.get('cve_description'))
    if raw_desc:
        return raw_desc

    cve_id = row.get('cve_id')
    if not cve_id or not str(cve_id).strip():
        return None
    prompt = cve_utils.get_prompt(str(cve_id).strip())
    if "was not found in the CVE dataset" in prompt:
        return None
    return prompt

df['cve_desc'] = df.apply(get_cve_desc, axis=1)

filled   = df['cve_desc'].notna().sum()
from_raw = df['cve_description'].apply(_normalize_cve_desc).notna().sum()
print(f"总样本数:                  {len(df)}")
print(f"cve_desc 已填充:           {filled}")
print(f"  └─ 来自 cve_description: {from_raw}")
print(f"  └─ 来自 CVEUtils 补充:   {filled - from_raw}")
print(f"cve_desc 为空:             {len(df) - filled}")
print(f"cve_desc 填充率:           {filled / len(df) * 100:.2f}%")
df[['cve_id', 'cve_desc']].head()

总样本数:                  1129
cve_desc 已填充:           1129
  └─ 来自 cve_description: 1129
  └─ 来自 CVEUtils 补充:   0
cve_desc 为空:             0
cve_desc 填充率:           100.00%


,cve_id,cve_desc
0,CVE-2011-0990,Race condition in the FastCopy optimization in...
1,CVE-2011-0991,"Use-after-free vulnerability in Mono, when Moo..."
2,CVE-2011-0991,"Use-after-free vulnerability in Mono, when Moo..."
3,CVE-2011-2517,Multiple buffer overflows in net/wireless/nl80...
4,CVE-2011-2518,The tomoyo_mount_acl function in security/tomo...


In [15]:
# ── language 属性：基于 file_language 扩展名 + detect_c_or_cpp 二次判定 ──

def determine_language(row):
    """根据 file_language 扩展名确定编程语言，头文件用 detect_c_or_cpp 二次判定。"""
    ext = str(row.get('file_language', '')).strip().lower()
    if ext in C_EXTENSIONS:
        return "C"
    if ext in CPP_EXTENSIONS:
        return "C++"
    if ext in HEADER_EXTENSIONS:
        result = detect_c_or_cpp(row.get('vul_code', ''))
        return "C" if result.language == "c" else "C++"
    return "C"

df['language'] = df.apply(determine_language, axis=1)

lang_counts = df['language'].value_counts()
print("language 分布:")
for lang, cnt in lang_counts.items():
    print(f"  {lang}: {cnt}")

# ── diff 属性 ──

def compute_diff(vul_code: str, fixed_code: str) -> str:
    """使用 difflib.unified_diff 计算函数级代码变更差异。"""
    a = vul_code.splitlines(keepends=True)
    b = fixed_code.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(a, b, fromfile="vul_code", tofile="fixed_code"))
    return "".join(diff_lines) if diff_lines else ""

df['diff'] = df.apply(lambda r: compute_diff(r['vul_code'], r['fixed_code']), axis=1)

empty_diff = (df['diff'] == '').sum()
print(f"\n总样本数:         {len(df)}")
print(f"diff 非空:        {len(df) - empty_diff}")
print(f"diff 为空:        {empty_diff}")
df[['language', 'vul_code', 'fixed_code', 'diff']].head(1)

language 分布:
  C: 1005
  C++: 124

总样本数:         1129
diff 非空:        1129
diff 为空:        0


,language,vul_code,fixed_code,diff
0,C,static gboolean ves_icall_System_Array_FastCop...,static gboolean ves_icall_System_Array_FastCop...,"--- vul_code\n+++ fixed_code\n@@ -6,7 +6,6 @@\..."


In [16]:
df = df.reset_index(drop=True)
df['id']     = df.index
df['source'] = 'ReposVul'

print(f"id 范围:   0 ~ {df['id'].max()}")
print(f"source:    {df['source'].unique().tolist()}")
print(f"总样本数:  {len(df)}")

# ── 导出 ──
FINAL_COLS = [
    'id', 'source', 'language',
    'cwe_id', 'cve_id',
    'cwe_desc', 'cve_desc',
    'vul_code', 'fixed_code',
    'diff',
]

df_out = df[FINAL_COLS].copy()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_out.to_json(
    OUTPUT_PATH,
    orient='records',
    lines=True,
    force_ascii=False,
)

print(f"\n导出完成: {OUTPUT_PATH}")
print(f"总样本数: {len(df_out)}")
print(f"\n字段统计：")
for col in FINAL_COLS:
    null_count = df_out[col].isna().sum()
    fill_rate  = (1 - null_count / len(df_out)) * 100
    print(f"  {col:<14}: 填充率 {fill_rate:6.2f}%  (空值 {null_count} 条)")
df_out.head()

id 范围:   0 ~ 1128
source:    ['ReposVul']
总样本数:  1129

导出完成: <PROJECT_ROOT>/data/raw/vul/ReposVul_Merged_v1.jsonl
总样本数: 1129

字段统计：
  id            : 填充率 100.00%  (空值 0 条)
  source        : 填充率 100.00%  (空值 0 条)
  language      : 填充率 100.00%  (空值 0 条)
  cwe_id        : 填充率 100.00%  (空值 0 条)
  cve_id        : 填充率 100.00%  (空值 0 条)
  cwe_desc      : 填充率  97.43%  (空值 29 条)
  cve_desc      : 填充率 100.00%  (空值 0 条)
  vul_code      : 填充率 100.00%  (空值 0 条)
  fixed_code    : 填充率 100.00%  (空值 0 条)
  diff          : 填充率 100.00%  (空值 0 条)


,id,source,language,cwe_id,cve_id,cwe_desc,cve_desc,vul_code,fixed_code,diff
0,0,ReposVul,C,CWE-362,CVE-2011-0990,- CWE ID: CWE-362\n- Name: Concurrent Executio...,Race condition in the FastCopy optimization in...,static gboolean ves_icall_System_Array_FastCop...,static gboolean ves_icall_System_Array_FastCop...,"--- vul_code\n+++ fixed_code\n@@ -6,7 +6,6 @@\..."
1,1,ReposVul,C,CWE-399,CVE-2011-0991,- CWE ID: CWE-399\n- Name: Resource Management...,"Use-after-free vulnerability in Mono, when Moo...",void mono_runtime_shutdown(void) {\n mono_dom...,void mono_runtime_shutdown(void) {\n mono_dom...,"--- vul_code\n+++ fixed_code\n@@ -1,3 +1,6 @@\..."
2,2,ReposVul,C,CWE-399,CVE-2011-0991,- CWE ID: CWE-399\n- Name: Resource Management...,"Use-after-free vulnerability in Mono, when Moo...",void mono_reflection_destroy_dynamic_method(Mo...,gboolean mono_reflection_is_valid_dynamic_toke...,"--- vul_code\n+++ fixed_code\n@@ -1,13 +1,3 @@..."
3,3,ReposVul,C,CWE-119,CVE-2011-2517,- CWE ID: CWE-119\n- Name: Improper Restrictio...,Multiple buffer overflows in net/wireless/nl80...,static int nl80211_trigger_scan(struct sk_buff...,static int nl80211_trigger_scan(struct sk_buff...,"--- vul_code\n+++ fixed_code\n@@ -113,12 +113,..."
4,4,ReposVul,C,CWE-20,CVE-2011-2518,- CWE ID: CWE-20\n- Name: Improper Input Valid...,The tomoyo_mount_acl function in security/tomo...,static int tomoyo_mount_acl(struct tomoyo_requ...,static int tomoyo_mount_acl(struct tomoyo_requ...,"--- vul_code\n+++ fixed_code\n@@ -50,7 +50,7 @..."
